In [6]:
!pip install tensorflow opencv-python



[notice] A new release of pip is available: 23.3.1 -> 24.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import cv2
import numpy as np
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from sklearn.model_selection import train_test_split

# Load and preprocess the dataset
root_dir = 'C:/Users/duaas/Downloads/QC/'
damaged_side_dir = os.path.join(root_dir, 'damaged/side/')
damaged_top_dir = os.path.join(root_dir, 'damaged/top/')
intact_side_dir = os.path.join(root_dir, 'intact/side/')
intact_top_dir = os.path.join(root_dir, 'intact/top/')

def load_images(directory):
    images = []
    for filename in os.listdir(directory):
        if filename.endswith('.jpg') or filename.endswith('.png'):
            img_path = os.path.join(directory, filename)
            img = cv2.imread(img_path)
            img = cv2.resize(img, (224, 224))  # Resize images to the input size of VGG16
            img = img.astype('float32') / 255.0  # Normalize pixel values
            images.append(img)
    return images

damaged_images = load_images(damaged_side_dir) + load_images(damaged_top_dir)
intact_images = load_images(intact_side_dir) + load_images(intact_top_dir)

X = np.array(damaged_images + intact_images)
y = np.array([1] * len(damaged_images) + [0] * len(intact_images))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Load pre-trained VGG16 model without top layers
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the convolutional base
base_model.trainable = False

# Build the model
model = Sequential([
    base_model,
    Flatten(),
    Dense(256, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.1)

# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {test_loss}, Test Accuracy: {test_accuracy}")


C:\Users\duaas\AppData\Local\Programs\Python\Python310\lib\site-packages\scipy\__init__.py:177: UserWarning: A NumPy version >=1.18.5 and <1.26.0 is required for this version of SciPy (detected version 1.26.2
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"






Epoch 1/20


9/9 [==============================] - 22s 2s/step - loss: 2.2979 - accuracy: 0.5451 - val_loss: 1.0707 - val_accuracy: 0.5312
Epoch 2/20
9/9 [==============================] - 18s 2s/step - loss: 0.8119 - accuracy: 0.5729 - val_loss: 0.8676 - val_accuracy: 0.5625
Epoch 3/20
9/9 [==============================] - 18s 2s/step - loss: 0.5884 - accuracy: 0.7153 - val_loss: 0.7738 - val_accuracy: 0.4688
Epoch 4/20
9/9 [==============================] - 18s 2s/step - loss: 0.5220 - accuracy: 0.7535 - val_loss: 0.7645 - val_accuracy: 0.5938
Epoch 5/20
9/9 [==============================] - 18s 2s/step - loss: 0.4691 - accuracy: 0.7743 - val_loss: 0.7831 - val_accuracy: 0.5312
Epoch 6/20
9/9 [==============================] - 18s 2s/step - loss: 0.4326 - accuracy: 0.8229 - val_loss: 0.8120 - val_accuracy: 0.4688
Epoch 7/20
9/9 [==============================] - 18s 2s/step - loss: 0.3766 - accuracy: 0.8889 - val_loss: 0.8608 - val_accuracy: 0.4688
Epoch 8/20
9/9 [============

In [9]:
# Save the trained model
model.save('vgg16_model.h5')


C:\Users\duaas\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [10]:
import tkinter as tk
from tkinter import filedialog
from PIL import Image, ImageTk
import numpy as np
from tensorflow.keras.models import load_model

# Load the saved model
model = load_model('vgg16_model.h5')

# Define class names
class_names = ['intact', 'damaged']

# Function to classify image
def classify_image(image_path):
    img = Image.open(image_path)
    img = img.resize((224, 224))  # Resize image to match VGG16 input size
    img = np.array(img) / 255.0    # Normalize pixel values
    img = np.expand_dims(img, axis=0)  # Add batch dimension

    # Perform classification
    prediction = model.predict(img)
    predicted_class = class_names[np.argmax(prediction)]
    return predicted_class

# Function to open file dialog and display selected image
def select_image():
    file_path = filedialog.askopenfilename()
    if file_path:
        # Display selected image
        img = Image.open(file_path)
        img.thumbnail((400, 400))
        img = ImageTk.PhotoImage(img)
        panel.configure(image=img)
        panel.image = img

        # Classify image and display result
        result = classify_image(file_path)
        result_label.configure(text=f"Predicted class: {result}")

# Create GUI window
window = tk.Tk()
window.title("Medicine Package Classification")

# Create file select button
select_button = tk.Button(window, text="Select Image", command=select_image)
select_button.pack(pady=10)

# Create panel to display image
panel = tk.Label(window)
panel.pack()

# Create label to display classification result
result_label = tk.Label(window, text="")
result_label.pack(pady=10)

# Run the GUI
window.mainloop()


1/1 [==============================] - 0s 94ms/step
